In [ ]:
from math import ceil, sqrt
from pathlib import Path
import shutil

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


In [ ]:
INPUT_DIR = "data/26Mar25-PyramidForcingView/videos/pyramid-forcing-60s"
OUTPUT_DIR = "figures/26Mar25-PyramidForcingView/pyramid-forcing-60s"
FRAME_STRIDE = 1

In [ ]:
VIDEO_SUFFIXES = (".mp4", ".mov", ".avi", ".mkv", ".webm")
FRAME_FILENAME_TEMPLATE = "frame_{frame_idx:06d}.png"
SHEET_ROWS = 6
SHEET_COLS = 6
SHEET_DPI = 200
SHEET_FIGSIZE = (14, 14)
SHEET_FACE_COLOR = "white"
SHEET_OVERVIEW_MAX_COLS = 4
SUMMARY_FILENAME = "frame_export_summary.csv"


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


def resolve_path(path_str: str) -> Path:
    raw = Path(path_str)
    return raw if raw.is_absolute() else resolve_repo_root() / raw


INPUT_PATH = resolve_path(INPUT_DIR)
OUTPUT_PATH = resolve_path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = OUTPUT_PATH / SUMMARY_FILENAME
{
    "input_dir": str(INPUT_PATH),
    "output_dir": str(OUTPUT_PATH),
    "frame_stride": FRAME_STRIDE,
}


In [ ]:
def list_local_videos(input_path: Path) -> list[Path]:
    videos = sorted(
        path
        for path in input_path.iterdir()
        if path.is_file() and path.suffix.lower() in VIDEO_SUFFIXES
    )
    if not videos:
        raise FileNotFoundError(f"No local videos found in: {input_path}")
    return videos


def normalize_video_id(video_path: Path, index: int) -> str:
    group_name = video_path.parent.name
    return f"{group_name}_video_{index:02d}"


def chunked(items: list[Path], chunk_size: int) -> list[list[Path]]:
    return [items[idx : idx + chunk_size] for idx in range(0, len(items), chunk_size)]


def render_image_grid(image_paths: list[Path], output_path: Path, rows: int, cols: int, title: str) -> None:
    fig, axes = plt.subplots(rows, cols, figsize=SHEET_FIGSIZE, dpi=SHEET_DPI)
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
    fig.patch.set_facecolor(SHEET_FACE_COLOR)
    fig.suptitle(title, fontsize=14)

    for axis, image_path in zip(axes, image_paths):
        axis.imshow(plt.imread(image_path))
        axis.axis("off")

    for axis in axes[len(image_paths) :]:
        axis.axis("off")

    fig.tight_layout()
    fig.savefig(output_path, dpi=SHEET_DPI, facecolor=SHEET_FACE_COLOR, bbox_inches="tight")
    plt.close(fig)


def render_sheet_pages(normalized_video_id: str, frame_paths: list[Path], video_dir: Path) -> list[Path]:
    pages = chunked(frame_paths, SHEET_ROWS * SHEET_COLS)
    sheet_paths = []
    for page_index, page_frame_paths in enumerate(pages, start=1):
        sheet_path = video_dir / f"sheet_{page_index:03d}.png"
        render_image_grid(
            page_frame_paths,
            sheet_path,
            rows=SHEET_ROWS,
            cols=SHEET_COLS,
            title=f"{normalized_video_id} | page {page_index}/{len(pages)}",
        )
        sheet_paths.append(sheet_path)
    return sheet_paths


def render_sheet_overview(normalized_video_id: str, sheet_paths: list[Path], video_dir: Path) -> Path:
    overview_path = video_dir / "sheet_overview.png"
    if len(sheet_paths) == 1:
        shutil.copy2(sheet_paths[0], overview_path)
        return overview_path

    overview_cols = min(SHEET_OVERVIEW_MAX_COLS, max(1, ceil(sqrt(len(sheet_paths)))))
    overview_rows = ceil(len(sheet_paths) / overview_cols)
    render_image_grid(
        sheet_paths,
        overview_path,
        rows=overview_rows,
        cols=overview_cols,
        title=f"{normalized_video_id} | sheet overview",
    )
    return overview_path


def export_video_frames(video_path: Path, index: int) -> dict:
    normalized_video_id = normalize_video_id(video_path, index)
    video_dir = OUTPUT_PATH / normalized_video_id
    frames_dir = video_dir / "frames"
    video_dir.mkdir(parents=True, exist_ok=True)
    frames_dir.mkdir(parents=True, exist_ok=True)

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        capture.release()
        raise RuntimeError(f"Could not open video with OpenCV: {video_path}")

    decoded_frame_count = 0
    frame_paths = []
    while True:
        ok, frame_bgr = capture.read()
        if not ok:
            break
        decoded_frame_count += 1
        if (decoded_frame_count - 1) % FRAME_STRIDE != 0:
            continue
        frame_output_path = frames_dir / FRAME_FILENAME_TEMPLATE.format(frame_idx=decoded_frame_count)
        if not cv2.imwrite(str(frame_output_path), frame_bgr):
            capture.release()
            raise RuntimeError(f"OpenCV failed to write frame PNG: {frame_output_path}")
        frame_paths.append(frame_output_path)

    capture.release()
    if not frame_paths:
        raise RuntimeError(f"No frames were exported for {video_path.name}")

    sheet_paths = render_sheet_pages(normalized_video_id, frame_paths, video_dir)
    overview_path = render_sheet_overview(normalized_video_id, sheet_paths, video_dir)
    return {
        "normalized_video_id": normalized_video_id,
        "video_path": str(video_path),
        "decoded_frame_count": decoded_frame_count,
        "exported_frame_count": len(frame_paths),
        "frame_stride": FRAME_STRIDE,
        "frame_dir": str(frames_dir),
        "sheet_count": len(sheet_paths),
        "overview_path": str(overview_path),
        "status": "ok",
    }


In [ ]:
video_paths = list_local_videos(INPUT_PATH)
summary_df = pd.DataFrame(
    [export_video_frames(video_path, index) for index, video_path in enumerate(video_paths, start=1)]
)
summary_df.to_csv(SUMMARY_PATH, index=False)
display(summary_df)
print(f"Saved frame export summary to {SUMMARY_PATH}")
